## Step 1: Data Loading

In [ ]:
import os
import numpy as np
from PIL import Image
import torch
from torchvision import datasets, transforms
 
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])
dataset = datasets.ImageFolder(root='dataset/', transform=transform)
data_loader = torch.utils.data.DataLoader(dataset, batch_size=16, shuffle=False)

## Step2: Feature Extraction

In [ ]:
import psycopg2
from psycopg2.extras import execute_values
from tqdm import tqdm

conn = psycopg2.connect(
    dbname="facedb",
    user="postgres",
    password="postgres",
    host="localhost",
    port=5432
)
cur = conn.cursor()

cur.execute("""
CREATE TABLE IF NOT EXISTS image_features (
    id SERIAL PRIMARY KEY,
    batch_id SMALLINT,
    image_id SMALLINT,
    channel_id SMALLINT,
    y_pos SMALLINT,
    x_pos SMALLINT,
    pixel_value REAL
)""")
conn.commit()

def extract_and_store_features(loader, sample_ratio=0.1):
    insert_query = "INSERT INTO image_features (batch_id, image_id, channel_id, y_pos, x_pos, pixel_value) VALUES %s"
    sample_step = max(1, int(1/sample_ratio))
    total_features = 0
    
    for batch_idx, (images, labels) in enumerate(tqdm(loader)):
        batch_images = images.numpy()
        batch_size, channels, height, width = batch_images.shape
        
        batch_data = []
        for img_idx in range(batch_size):
            for c in range(channels):
                for y in range(0, height, sample_step):
                    for x in range(0, width, sample_step):
                        pixel_val = round(float(batch_images[img_idx, c, y, x]), 4)
                        if abs(pixel_val) > 0.001:
                            batch_data.append((batch_idx, img_idx, c, y, x, pixel_val))
        
        if batch_data:
            execute_values(cur, insert_query, batch_data)
            conn.commit()
            total_features += len(batch_data)
            print(f"Batch {batch_idx}: stored {len(batch_data)} feature points")
    
    return total_features

sample_ratio = 0.1
total_stored = extract_and_store_features(data_loader, sample_ratio)
print(f"Total stored: {total_stored} points")

cur.close()
conn.close()


## Step 3: Model Profiling

In [ ]:
import onnx
import time
import onnxruntime as ort

model = onnx.load("resnetx50.onnx")
session = ort.InferenceSession("resnext50.onnx")
dummy_input = np.random.randn(1, 3, 224, 224).astype(np.float32)
input_name = session.get_inputs()[0].name
start = time.time()
for _ in range(50):
    _ = session.run(None, {input_name: dummy_input})
print(f"Parameters: {len(model.graph.initializer)}, Time: {(time.time()-start)/50*1000:.2f}ms")

## Step 4: Data Preparation

In [ ]:
import onnx
import numpy as np
import psycopg2
from psycopg2.extras import execute_values
import time

class ModelParamConverter:
    def __init__(self, db_config):
        self.conn = psycopg2.connect(**db_config)
        self.cur = self.conn.cursor()
        self.drop_records = []
    
    def close(self):
        self.cur.close()
        self.conn.close()
    
    def write_command(self, command):
        self.cur.execute(command)
        self.conn.commit()
    
    def write_batch_data(self, query, data):
        execute_values(self.cur, query, data)
        self.conn.commit()
    
    def profile_model(self, model_path, input_shape=(1, 3, 224, 224), num_runs=10):
        model = onnx.load(model_path)
        print(f"Model loaded: {model_path}")
        print(f"Nodes: {len(model.graph.node)}")
        print(f"Parameters: {len(model.graph.initializer)}")
        
        return model
    
    def trans_params(self, table_mapping_tensors):
        create_queries = []
        insert_queries = []
        delete_quereis = []
        db_array_mappings = {}
        
        for name, vals in table_mapping_tensors.items():
            val = vals[0]
            if len(val.shape) == 2:
                create_q = f"CREATE TABLE IF NOT EXISTS {name} (kernel_id smallint, order_id smallint, value real)"
                insert_q = f"INSERT INTO {name} (kernel_id, order_id, value) VALUES (%s, %s, %s)"
                batch_data = []
                for kernel_id in range(val.shape[0]):
                    for order_id in range(val.shape[1]):
                        batch_data.append((kernel_id, order_id, float(val[kernel_id][order_id])))
            else:
                create_q = f"CREATE TABLE IF NOT EXISTS {name} (gn smallint, kernel_id smallint, order_id smallint, value real)"
                insert_q = f"INSERT INTO {name} (gn, kernel_id, order_id, value) VALUES (%s, %s, %s, %s)"
                batch_data = []
                for gn in range(val.shape[0]):
                    for kernel_id in range(val.shape[1]):
                        for order_id in range(val.shape[2]):
                            batch_data.append((gn, kernel_id, order_id, float(val[gn][kernel_id][order_id])))
                            
            db_array_mappings[name] = batch_data
            create_queries.append(create_q)
            insert_queries.append((insert_q, name))
            drop_q = f"DROP TABLE IF EXISTS {name};"
            delete_quereis.append(drop_q)
            
        for create_q in create_queries:
            self.write_command(create_q)
         
        for insert_q in insert_queries:
            self.write_batch_data(insert_q[0], db_array_mappings[insert_q[1]])
        
        with open("drop.sql", 'a+') as file:
            file.write("\n".join(delete_quereis))
            file.write("\n")
            
    def generate_mapping_table(self, mapping_name, w, kernel_size, stride, padding, channel):
        existed_tables = ["Relu189_fwd0_mapping","Relu211_fwd0_mapping"]
        if mapping_name in existed_tables:
            return
            
        h = w 
        batch_data = []
        create_q = f"CREATE TABLE IF NOT EXISTS {mapping_name} (kernel_id smallint, tuple_id smallint, matrix_id smallint, order_id smallint)"
        insert_q = f"INSERT INTO {mapping_name} (kernel_id, tuple_id, matrix_id, order_id) VALUES (%s, %s, %s, %s)"
        
        matrix_id = 0
        for y in range(0, h + 2*padding - kernel_size + 1, stride):
            for x in range(0, w + 2*padding - kernel_size + 1, stride):
                order_id = 0
                for c in range(channel):
                    for cy in range(y, y + kernel_size):
                        for cx in range(x, x + kernel_size):
                            if cy >= padding and cy < h+padding and cx >= padding and cx < w + padding:
                                tuple_id = (cy-padding) * w + cx - padding 
                            else:
                                tuple_id = -1
                            if tuple_id != -1:
                                batch_data.append((c, tuple_id, matrix_id, order_id))
                            order_id += 1
                matrix_id+= 1
                
        self.write_command(create_q)
        self.write_batch_data(insert_q, batch_data)
        self.drop_records.append(mapping_name)
        
    def generate_group_mapping_table(self, mapping_name, w, kernel_size, stride, padding, channel, groups):
        existed_tables = ["Relu170_fwd0_mapping", "Relu177_fwd0_mapping","Relu184_fwd0_mapping",
                        "Relu192_fwd0_mapping", "Relu199_fwd0_mapping", "Relu206_fwd0_mapping",
                        "Relu214_fwd0_mapping", "Relu221_fwd0_mapping", "Relu228_fwd0_mapping"]
        if mapping_name in existed_tables:
            return
            
        h = w 
        batch_data = []
        create_q = f"CREATE TABLE IF NOT EXISTS {mapping_name} (gn smallint, kernel_id smallint, tuple_id smallint, matrix_id smallint, order_id smallint)"
        insert_q = f"INSERT INTO {mapping_name} (gn, kernel_id, tuple_id, matrix_id, order_id) VALUES (%s, %s, %s, %s, %s)"
        
        channel_mapping_gn = [0] * channel
        channel_per_group = channel // groups
        for c in range(channel):
            channel_mapping_gn[c] = c // channel_per_group
            
        matrix_id = 0
        sizes_per_group = kernel_size * kernel_size * (channel//groups)
        for y in range(0, h + 2*padding - kernel_size + 1, stride):
            for x in range(0, w + 2*padding - kernel_size + 1, stride):
                order_id = 0
                for c in range(channel):
                    for cy in range(y, y + kernel_size):
                        for cx in range(x, x + kernel_size):
                            if cy >= padding and cy < h+padding and cx >= padding and cx < w + padding:
                                tuple_id = (cy-padding) * w + cx - padding 
                            else:
                                tuple_id = -1
                            if tuple_id != -1:
                                batch_data.append((channel_mapping_gn[c], c, tuple_id, matrix_id, order_id%sizes_per_group))
                            order_id += 1
                matrix_id+= 1
                
        self.write_command(create_q)
        self.write_batch_data(insert_q, batch_data)
        self.drop_records.append(mapping_name)
        
    def extract_model_parameters(self, model_path):
        model = self.profile_model(model_path)
        
         
        table_mapping_tensors = {}
        for tensor in model.graph.initializer:
            
            tensor_name = tensor.name
            
            tensor_shape = tuple(dim for dim in tensor.dims)
            np_array = None
            if tensor.data_type == 1:  # FLOAT
                np_array = np.frombuffer(tensor.raw_data, dtype=np.float32).reshape(tensor_shape)
            
            if np_array is not None:
                table_name = f"{tensor_name.replace('.', '_')}_table"
                table_mapping_tensors[table_name] = [np_array]
                
                if len(tensor_shape) > 1:  
                    print(f"Processing: {tensor_name} -> {table_name}, Shape: {tensor_shape}")
        
        print(f"Found {len(table_mapping_tensors)} parameter tables")
        return table_mapping_tensors
        
    def convert_model_to_db(self, model_path):
        start_time = time.time()
        table_mapping_tensors = self.extract_model_parameters(model_path)
        
        self.trans_params(table_mapping_tensors)
        
        self.generate_mapping_table("conv1_mapping", 224, 7, 2, 3, 3)
        self.generate_mapping_table("layer1_mapping", 56, 3, 1, 1, 64)
        
        
        print(f"Model conversion completed in {time.time() - start_time:.2f} seconds")
        print(f"Generated {len(self.drop_records)} mapping tables")

db_config = {
    "dbname": "facedb", 
    "user": "postgres", 
    "password": "postgres", 
    "host": "localhost"
}

converter = ModelParamConverter(db_config)
try:
    converter.convert_model_to_db("resnetx50.onnx")
finally:
    converter.close()


## Step 5: Query Composition

In [1]:
'''
WITH
Conv201_fwd0 AS (
	select batch_id, K.kernel_id, F.matrix_id as tuple_id,
	sum(F.value*K.value) as value
		FROM input_feature_map F INNER JOIN Conv201_weight K
		on F.order_id=K.order_id
	GROUP BY F.batch_id, K.kernel_id, F.matrix_id
	),
Relu203_fwd0 AS (
	select batch_id, kernel_id, tuple_id, GREATEST(value, 0) as value
		FROM Conv201_fwd0 F
	),
Relu203_fwd0_mapped AS (
	select batch_id, F.kernel_id, matrix_id, order_id, value
		FROM Relu203_fwd0 F INNER JOIN Relu203_fwd0_mapping M
		on F.kernel_id = M.kernel_id and F.tuple_id = M.tuple_id
	),
MaxPool205_fwd0 AS (
	select batch_id, kernel_id, matrix_id as tuple_id, max(value) as value
		FROM Relu203_fwd0_mapped
	GROUP BY batch_id, kernel_id, matrix_id
	),
Conv206_fwd0 AS (
	select batch_id, K.kernel_id, F.tuple_id as tuple_id,
	sum(F.value*K.value) as value
		FROM MaxPool205_fwd0 F INNER JOIN Conv206_weight K
		on F.kernel_id=K.order_id
	GROUP BY F.batch_id, K.kernel_id, F.tuple_id
	),
Conv207_fwd0 AS (
	select batch_id, K.kernel_id, F.tuple_id as tuple_id,
	sum(F.value*K.value) as value
		FROM MaxPool205_fwd0 F INNER JOIN Conv207_weight K
		on F.kernel_id=K.order_id
	GROUP BY F.batch_id, K.kernel_id, F.tuple_id
	),
Relu208_fwd0 AS (
	select batch_id, kernel_id, tuple_id, GREATEST(value, 0) as value
		FROM Conv206_fwd0 F
	),
Relu208_fwd0_mapped AS (
	select batch_id, gn, matrix_id, order_id, value
		FROM Relu208_fwd0 F INNER JOIN Relu208_fwd0_mapping M
		on F.kernel_id = M.kernel_id and F.tuple_id = M.tuple_id
	),
Conv209_fwd0 AS (
	select batch_id, (F.gn*4+kernel_id) as kernel_id, F.matrix_id as tuple_id,
	sum(F.value*K.value) as value
		FROM Relu208_fwd0_mapped F INNER JOIN Conv209_weight K
		on F.order_id=K.order_id and F.gn=K.gn
	GROUP BY F.batch_id, F.gn, K.kernel_id, F.matrix_id
	),
Relu210_fwd0 AS (
	select batch_id, kernel_id, tuple_id, GREATEST(value, 0) as value
		FROM Conv209_fwd0 F
	),
Conv211_fwd0 AS (
	select batch_id, K.kernel_id, F.tuple_id as tuple_id,
	sum(F.value*K.value) as value
		FROM Relu210_fwd0 F INNER JOIN Conv211_weight K
		on F.kernel_id=K.order_id
	GROUP BY F.batch_id, K.kernel_id, F.tuple_id
	),
Add212_fwd0 AS (
	select A.batch_id, A.kernel_id, A.tuple_id,
	A.value + B.value as value
		FROM Conv207_fwd0 A INNER JOIN Conv211_fwd0 B
		on A.batch_id=B.batch_id and A.kernel_id=B.kernel_id
		and A.tuple_id=B.tuple_id
	),
Relu213_fwd0 AS (
	select batch_id, kernel_id, tuple_id, GREATEST(value, 0) as value
		FROM Add212_fwd0 F
	),
Conv214_fwd0 AS (
	select batch_id, K.kernel_id, F.tuple_id as tuple_id,
	sum(F.value*K.value) as value
		FROM Relu213_fwd0 F INNER JOIN Conv214_weight K
		on F.kernel_id=K.order_id
	GROUP BY F.batch_id, K.kernel_id, F.tuple_id
	),
Relu215_fwd0 AS (
	select batch_id, kernel_id, tuple_id, GREATEST(value, 0) as value
		FROM Conv214_fwd0 F
	),
Relu215_fwd0_mapped AS (
	select batch_id, gn, matrix_id, order_id, value
		FROM Relu215_fwd0 F INNER JOIN Relu215_fwd0_mapping M
		on F.kernel_id = M.kernel_id and F.tuple_id = M.tuple_id
	),
Conv216_fwd0 AS (
	select batch_id, (F.gn*4+kernel_id) as kernel_id, F.matrix_id as tuple_id,
	sum(F.value*K.value) as value
		FROM Relu215_fwd0_mapped F INNER JOIN Conv216_weight K
		on F.order_id=K.order_id and F.gn=K.gn
	GROUP BY F.batch_id, F.gn, K.kernel_id, F.matrix_id
	),
Relu217_fwd0 AS (
	select batch_id, kernel_id, tuple_id, GREATEST(value, 0) as value
		FROM Conv216_fwd0 F
	),
Conv218_fwd0 AS (
	select batch_id, K.kernel_id, F.tuple_id as tuple_id,
	sum(F.value*K.value) as value
		FROM Relu217_fwd0 F INNER JOIN Conv218_weight K
		on F.kernel_id=K.order_id
	GROUP BY F.batch_id, K.kernel_id, F.tuple_id
	),
Add219_fwd0 AS (
	select A.batch_id, A.kernel_id, A.tuple_id,
	A.value + B.value as value
		FROM Relu213_fwd0 A INNER JOIN Conv218_fwd0 B
		on A.batch_id=B.batch_id and A.kernel_id=B.kernel_id
		and A.tuple_id=B.tuple_id
	),
Relu220_fwd0 AS (
	select batch_id, kernel_id, tuple_id, GREATEST(value, 0) as value
		FROM Add219_fwd0 F
	),
Conv221_fwd0 AS (
	select batch_id, K.kernel_id, F.tuple_id as tuple_id,
	sum(F.value*K.value) as value
		FROM Relu220_fwd0 F INNER JOIN Conv221_weight K
		on F.kernel_id=K.order_id
	GROUP BY F.batch_id, K.kernel_id, F.tuple_id
	),
Relu222_fwd0 AS (
	select batch_id, kernel_id, tuple_id, GREATEST(value, 0) as value
		FROM Conv221_fwd0 F
	),
Relu222_fwd0_mapped AS (
	select batch_id, gn, matrix_id, order_id, value
		FROM Relu222_fwd0 F INNER JOIN Relu222_fwd0_mapping M
		on F.kernel_id = M.kernel_id and F.tuple_id = M.tuple_id
	),
Conv223_fwd0 AS (
	select batch_id, (F.gn*4+kernel_id) as kernel_id, F.matrix_id as tuple_id,
	sum(F.value*K.value) as value
		FROM Relu222_fwd0_mapped F INNER JOIN Conv223_weight K
		on F.order_id=K.order_id and F.gn=K.gn
	GROUP BY F.batch_id, F.gn, K.kernel_id, F.matrix_id
	),
Relu224_fwd0 AS (
	select batch_id, kernel_id, tuple_id, GREATEST(value, 0) as value
		FROM Conv223_fwd0 F
	),
Conv225_fwd0 AS (
	select batch_id, K.kernel_id, F.tuple_id as tuple_id,
	sum(F.value*K.value) as value
		FROM Relu224_fwd0 F INNER JOIN Conv225_weight K
		on F.kernel_id=K.order_id
	GROUP BY F.batch_id, K.kernel_id, F.tuple_id
	),
Add226_fwd0 AS (
	select A.batch_id, A.kernel_id, A.tuple_id,
	A.value + B.value as value
		FROM Relu220_fwd0 A INNER JOIN Conv225_fwd0 B
		on A.batch_id=B.batch_id and A.kernel_id=B.kernel_id
		and A.tuple_id=B.tuple_id
	),
Relu227_fwd0 AS (
	select batch_id, kernel_id, tuple_id, GREATEST(value, 0) as value
		FROM Add226_fwd0 F
	),
Conv228_fwd0 AS (
	select batch_id, K.kernel_id, F.tuple_id as tuple_id,
	sum(F.value*K.value) as value
		FROM Relu227_fwd0 F INNER JOIN Conv228_weight K
		on F.kernel_id=K.order_id
	GROUP BY F.batch_id, K.kernel_id, F.tuple_id
	),
Relu229_fwd0 AS (
	select batch_id, kernel_id, tuple_id, GREATEST(value, 0) as value
		FROM Conv228_fwd0 F
	),
Relu229_fwd0_mapped AS (
	select batch_id, gn, matrix_id, order_id, value
		FROM Relu229_fwd0 F INNER JOIN Relu229_fwd0_mapping M
		on F.kernel_id = M.kernel_id and F.tuple_id = M.tuple_id
	),
Conv230_fwd0 AS (
	select batch_id, (F.gn*4+kernel_id) as kernel_id, F.matrix_id as tuple_id,
	sum(F.value*K.value) as value
		FROM Relu229_fwd0_mapped F INNER JOIN Conv230_weight K
		on F.order_id=K.order_id and F.gn=K.gn
	GROUP BY F.batch_id, F.gn, K.kernel_id, F.matrix_id
	),
Relu231_fwd0 AS (
	select batch_id, kernel_id, tuple_id, GREATEST(value, 0) as value
		FROM Conv230_fwd0 F
	),
Conv232_fwd0 AS (
	select batch_id, K.kernel_id, F.tuple_id as tuple_id,
	sum(F.value*K.value) as value
		FROM Relu231_fwd0 F INNER JOIN Conv232_weight K
		on F.kernel_id=K.order_id
	GROUP BY F.batch_id, K.kernel_id, F.tuple_id
	),
Add233_fwd0 AS (
	select A.batch_id, A.kernel_id, A.tuple_id,
	A.value + B.value as value
		FROM Relu227_fwd0 A INNER JOIN Conv232_fwd0 B
		on A.batch_id=B.batch_id and A.kernel_id=B.kernel_id
		and A.tuple_id=B.tuple_id
	),
Relu234_fwd0 AS (
	select batch_id, kernel_id, tuple_id, GREATEST(value, 0) as value
		FROM Add233_fwd0 F
	),
Conv235_fwd0 AS (
	select batch_id, K.kernel_id, F.tuple_id as tuple_id,
	sum(F.value*K.value) as value
		FROM Relu234_fwd0 F INNER JOIN Conv235_weight K
		on F.kernel_id=K.order_id
	GROUP BY F.batch_id, K.kernel_id, F.tuple_id
	),
Relu236_fwd0 AS (
	select batch_id, kernel_id, tuple_id, GREATEST(value, 0) as value
		FROM Conv235_fwd0 F
	),
Relu236_fwd0_mapped AS (
	select batch_id, gn, matrix_id, order_id, value
		FROM Relu236_fwd0 F INNER JOIN Relu236_fwd0_mapping M
		on F.kernel_id = M.kernel_id and F.tuple_id = M.tuple_id
	),
Conv237_fwd0 AS (
	select batch_id, (F.gn*4+kernel_id) as kernel_id, F.matrix_id as tuple_id,
	sum(F.value*K.value) as value
		FROM Relu236_fwd0_mapped F INNER JOIN Conv237_weight K
		on F.order_id=K.order_id and F.gn=K.gn
	GROUP BY F.batch_id, F.gn, K.kernel_id, F.matrix_id
	),
Relu238_fwd0 AS (
	select batch_id, kernel_id, tuple_id, GREATEST(value, 0) as value
		FROM Conv237_fwd0 F
	),
Conv239_fwd0 AS (
	select batch_id, K.kernel_id, F.tuple_id as tuple_id,
	sum(F.value*K.value) as value
		FROM Relu238_fwd0 F INNER JOIN Conv239_weight K
		on F.kernel_id=K.order_id
	GROUP BY F.batch_id, K.kernel_id, F.tuple_id
	),
Add240_fwd0 AS (
	select A.batch_id, A.kernel_id, A.tuple_id,
	A.value + B.value as value
		FROM Relu234_fwd0 A INNER JOIN Conv239_fwd0 B
		on A.batch_id=B.batch_id and A.kernel_id=B.kernel_id
		and A.tuple_id=B.tuple_id
	),
Relu241_fwd0 AS (
	select batch_id, kernel_id, tuple_id, GREATEST(value, 0) as value
		FROM Add240_fwd0 F
	),
Conv242_fwd0 AS (
	select batch_id, K.kernel_id, F.tuple_id as tuple_id,
	sum(F.value*K.value) as value
		FROM Relu241_fwd0 F INNER JOIN Conv242_weight K
		on F.kernel_id=K.order_id
	GROUP BY F.batch_id, K.kernel_id, F.tuple_id
	),
Relu241_fwd0_mapped AS (
	select batch_id, matrix_id, order_id, value
		FROM Relu241_fwd0 F INNER JOIN Relu241_fwd0_mapping M
		on F.kernel_id = M.kernel_id and F.tuple_id = M.tuple_id
	),
Conv243_fwd0 AS (
	select batch_id, K.kernel_id, F.matrix_id as tuple_id,
	sum(F.value*K.value) as value
		FROM Relu241_fwd0_mapped F INNER JOIN Conv243_weight K
		on F.order_id=K.order_id
	GROUP BY F.batch_id, K.kernel_id, F.matrix_id
	),
Relu244_fwd0 AS (
	select batch_id, kernel_id, tuple_id, GREATEST(value, 0) as value
		FROM Conv242_fwd0 F
	),
Relu244_fwd0_mapped AS (
	select batch_id, gn, matrix_id, order_id, value
		FROM Relu244_fwd0 F INNER JOIN Relu244_fwd0_mapping M
		on F.kernel_id = M.kernel_id and F.tuple_id = M.tuple_id
	),
Conv245_fwd0 AS (
	select batch_id, (F.gn*8+kernel_id) as kernel_id, F.matrix_id as tuple_id,
	sum(F.value*K.value) as value
		FROM Relu244_fwd0_mapped F INNER JOIN Conv245_weight K
		on F.order_id=K.order_id and F.gn=K.gn
	GROUP BY F.batch_id, F.gn, K.kernel_id, F.matrix_id
	),
Relu246_fwd0 AS (
	select batch_id, kernel_id, tuple_id, GREATEST(value, 0) as value
		FROM Conv245_fwd0 F
	),
Conv247_fwd0 AS (
	select batch_id, K.kernel_id, F.tuple_id as tuple_id,
	sum(F.value*K.value) as value
		FROM Relu246_fwd0 F INNER JOIN Conv247_weight K
		on F.kernel_id=K.order_id
	GROUP BY F.batch_id, K.kernel_id, F.tuple_id
	),
Add248_fwd0 AS (
	select A.batch_id, A.kernel_id, A.tuple_id,
	A.value + B.value as value
		FROM Conv243_fwd0 A INNER JOIN Conv247_fwd0 B
		on A.batch_id=B.batch_id and A.kernel_id=B.kernel_id
		and A.tuple_id=B.tuple_id
	),
Relu249_fwd0 AS (
	select batch_id, kernel_id, tuple_id, GREATEST(value, 0) as value
		FROM Add248_fwd0 F
	),
Conv250_fwd0 AS (
	select batch_id, K.kernel_id, F.tuple_id as tuple_id,
	sum(F.value*K.value) as value
		FROM Relu249_fwd0 F INNER JOIN Conv250_weight K
		on F.kernel_id=K.order_id
	GROUP BY F.batch_id, K.kernel_id, F.tuple_id
	),
Relu251_fwd0 AS (
	select batch_id, kernel_id, tuple_id, GREATEST(value, 0) as value
		FROM Conv250_fwd0 F
	),
Relu251_fwd0_mapped AS (
	select batch_id, gn, matrix_id, order_id, value
		FROM Relu251_fwd0 F INNER JOIN Relu251_fwd0_mapping M
		on F.kernel_id = M.kernel_id and F.tuple_id = M.tuple_id
	),
Conv252_fwd0 AS (
	select batch_id, (F.gn*8+kernel_id) as kernel_id, F.matrix_id as tuple_id,
	sum(F.value*K.value) as value
		FROM Relu251_fwd0_mapped F INNER JOIN Conv252_weight K
		on F.order_id=K.order_id and F.gn=K.gn
	GROUP BY F.batch_id, F.gn, K.kernel_id, F.matrix_id
	),
Relu253_fwd0 AS (
	select batch_id, kernel_id, tuple_id, GREATEST(value, 0) as value
		FROM Conv252_fwd0 F
	),
Conv254_fwd0 AS (
	select batch_id, K.kernel_id, F.tuple_id as tuple_id,
	sum(F.value*K.value) as value
		FROM Relu253_fwd0 F INNER JOIN Conv254_weight K
		on F.kernel_id=K.order_id
	GROUP BY F.batch_id, K.kernel_id, F.tuple_id
	),
Add255_fwd0 AS (
	select A.batch_id, A.kernel_id, A.tuple_id,
	A.value + B.value as value
		FROM Relu249_fwd0 A INNER JOIN Conv254_fwd0 B
		on A.batch_id=B.batch_id and A.kernel_id=B.kernel_id
		and A.tuple_id=B.tuple_id
	),
Relu256_fwd0 AS (
	select batch_id, kernel_id, tuple_id, GREATEST(value, 0) as value
		FROM Add255_fwd0 F
	),
Conv257_fwd0 AS (
	select batch_id, K.kernel_id, F.tuple_id as tuple_id,
	sum(F.value*K.value) as value
		FROM Relu256_fwd0 F INNER JOIN Conv257_weight K
		on F.kernel_id=K.order_id
	GROUP BY F.batch_id, K.kernel_id, F.tuple_id
	),
Relu258_fwd0 AS (
	select batch_id, kernel_id, tuple_id, GREATEST(value, 0) as value
		FROM Conv257_fwd0 F
	),
Relu258_fwd0_mapped AS (
	select batch_id, gn, matrix_id, order_id, value
		FROM Relu258_fwd0 F INNER JOIN Relu258_fwd0_mapping M
		on F.kernel_id = M.kernel_id and F.tuple_id = M.tuple_id
	),
Conv259_fwd0 AS (
	select batch_id, (F.gn*8+kernel_id) as kernel_id, F.matrix_id as tuple_id,
	sum(F.value*K.value) as value
		FROM Relu258_fwd0_mapped F INNER JOIN Conv259_weight K
		on F.order_id=K.order_id and F.gn=K.gn
	GROUP BY F.batch_id, F.gn, K.kernel_id, F.matrix_id
	),
Relu260_fwd0 AS (
	select batch_id, kernel_id, tuple_id, GREATEST(value, 0) as value
		FROM Conv259_fwd0 F
	),
Conv261_fwd0 AS (
	select batch_id, K.kernel_id, F.tuple_id as tuple_id,
	sum(F.value*K.value) as value
		FROM Relu260_fwd0 F INNER JOIN Conv261_weight K
		on F.kernel_id=K.order_id
	GROUP BY F.batch_id, K.kernel_id, F.tuple_id
	),
Add262_fwd0 AS (
	select A.batch_id, A.kernel_id, A.tuple_id,
	A.value + B.value as value
		FROM Relu256_fwd0 A INNER JOIN Conv261_fwd0 B
		on A.batch_id=B.batch_id and A.kernel_id=B.kernel_id
		and A.tuple_id=B.tuple_id
	),
Relu263_fwd0 AS (
	select batch_id, kernel_id, tuple_id, GREATEST(value, 0) as value
		FROM Add262_fwd0 F
	),
Conv264_fwd0 AS (
	select batch_id, K.kernel_id, F.tuple_id as tuple_id,
	sum(F.value*K.value) as value
		FROM Relu263_fwd0 F INNER JOIN Conv264_weight K
		on F.kernel_id=K.order_id
	GROUP BY F.batch_id, K.kernel_id, F.tuple_id
	),
Relu265_fwd0 AS (
	select batch_id, kernel_id, tuple_id, GREATEST(value, 0) as value
		FROM Conv264_fwd0 F
	),
Relu265_fwd0_mapped AS (
	select batch_id, gn, matrix_id, order_id, value
		FROM Relu265_fwd0 F INNER JOIN Relu265_fwd0_mapping M
		on F.kernel_id = M.kernel_id and F.tuple_id = M.tuple_id
	),
Conv266_fwd0 AS (
	select batch_id, (F.gn*8+kernel_id) as kernel_id, F.matrix_id as tuple_id,
	sum(F.value*K.value) as value
		FROM Relu265_fwd0_mapped F INNER JOIN Conv266_weight K
		on F.order_id=K.order_id and F.gn=K.gn
	GROUP BY F.batch_id, F.gn, K.kernel_id, F.matrix_id
	),
Relu267_fwd0 AS (
	select batch_id, kernel_id, tuple_id, GREATEST(value, 0) as value
		FROM Conv266_fwd0 F
	),
Conv268_fwd0 AS (
	select batch_id, K.kernel_id, F.tuple_id as tuple_id,
	sum(F.value*K.value) as value
		FROM Relu267_fwd0 F INNER JOIN Conv268_weight K
		on F.kernel_id=K.order_id
	GROUP BY F.batch_id, K.kernel_id, F.tuple_id
	),
Add269_fwd0 AS (
	select A.batch_id, A.kernel_id, A.tuple_id,
	A.value + B.value as value
		FROM Relu263_fwd0 A INNER JOIN Conv268_fwd0 B
		on A.batch_id=B.batch_id and A.kernel_id=B.kernel_id
		and A.tuple_id=B.tuple_id
	),
Relu270_fwd0 AS (
	select batch_id, kernel_id, tuple_id, GREATEST(value, 0) as value
		FROM Add269_fwd0 F
	),
Conv271_fwd0 AS (
	select batch_id, K.kernel_id, F.tuple_id as tuple_id,
	sum(F.value*K.value) as value
		FROM Relu270_fwd0 F INNER JOIN Conv271_weight K
		on F.kernel_id=K.order_id
	GROUP BY F.batch_id, K.kernel_id, F.tuple_id
	),
Relu272_fwd0 AS (
	select batch_id, kernel_id, tuple_id, GREATEST(value, 0) as value
		FROM Conv271_fwd0 F
	),
Relu272_fwd0_mapped AS (
	select batch_id, gn, matrix_id, order_id, value
		FROM Relu272_fwd0 F INNER JOIN Relu272_fwd0_mapping M
		on F.kernel_id = M.kernel_id and F.tuple_id = M.tuple_id
	),
Conv273_fwd0 AS (
	select batch_id, (F.gn*8+kernel_id) as kernel_id, F.matrix_id as tuple_id,
	sum(F.value*K.value) as value
		FROM Relu272_fwd0_mapped F INNER JOIN Conv273_weight K
		on F.order_id=K.order_id and F.gn=K.gn
	GROUP BY F.batch_id, F.gn, K.kernel_id, F.matrix_id
	),
Relu274_fwd0 AS (
	select batch_id, kernel_id, tuple_id, GREATEST(value, 0) as value
		FROM Conv273_fwd0 F
	),
Conv275_fwd0 AS (
	select batch_id, K.kernel_id, F.tuple_id as tuple_id,
	sum(F.value*K.value) as value
		FROM Relu274_fwd0 F INNER JOIN Conv275_weight K
		on F.kernel_id=K.order_id
	GROUP BY F.batch_id, K.kernel_id, F.tuple_id
	),
Add276_fwd0 AS (
	select A.batch_id, A.kernel_id, A.tuple_id,
	A.value + B.value as value
		FROM Relu270_fwd0 A INNER JOIN Conv275_fwd0 B
		on A.batch_id=B.batch_id and A.kernel_id=B.kernel_id
		and A.tuple_id=B.tuple_id
	),
Relu277_fwd0 AS (
	select batch_id, kernel_id, tuple_id, GREATEST(value, 0) as value
		FROM Add276_fwd0 F
	),
Conv278_fwd0 AS (
	select batch_id, K.kernel_id, F.tuple_id as tuple_id,
	sum(F.value*K.value) as value
		FROM Relu277_fwd0 F INNER JOIN Conv278_weight K
		on F.kernel_id=K.order_id
	GROUP BY F.batch_id, K.kernel_id, F.tuple_id
	),
Relu277_fwd0_mapped AS (
	select batch_id, matrix_id, order_id, value
		FROM Relu277_fwd0 F INNER JOIN Relu277_fwd0_mapping M
		on F.kernel_id = M.kernel_id and F.tuple_id = M.tuple_id
	),
Conv279_fwd0 AS (
	select batch_id, K.kernel_id, F.matrix_id as tuple_id,
	sum(F.value*K.value) as value
		FROM Relu277_fwd0_mapped F INNER JOIN Conv279_weight K
		on F.order_id=K.order_id
	GROUP BY F.batch_id, K.kernel_id, F.matrix_id
	),
Relu280_fwd0 AS (
	select batch_id, kernel_id, tuple_id, GREATEST(value, 0) as value
		FROM Conv278_fwd0 F
	),
Relu280_fwd0_mapped AS (
	select batch_id, gn, matrix_id, order_id, value
		FROM Relu280_fwd0 F INNER JOIN Relu280_fwd0_mapping M
		on F.kernel_id = M.kernel_id and F.tuple_id = M.tuple_id
	),
Conv281_fwd0 AS (
	select batch_id, (F.gn*16+kernel_id) as kernel_id, F.matrix_id as tuple_id,
	sum(F.value*K.value) as value
		FROM Relu280_fwd0_mapped F INNER JOIN Conv281_weight K
		on F.order_id=K.order_id and F.gn=K.gn
	GROUP BY F.batch_id, F.gn, K.kernel_id, F.matrix_id
	),
Relu282_fwd0 AS (
	select batch_id, kernel_id, tuple_id, GREATEST(value, 0) as value
		FROM Conv281_fwd0 F
	),
Conv283_fwd0 AS (
	select batch_id, K.kernel_id, F.tuple_id as tuple_id,
	sum(F.value*K.value) as value
		FROM Relu282_fwd0 F INNER JOIN Conv283_weight K
		on F.kernel_id=K.order_id
	GROUP BY F.batch_id, K.kernel_id, F.tuple_id
	),
Add284_fwd0 AS (
	select A.batch_id, A.kernel_id, A.tuple_id,
	A.value + B.value as value
		FROM Conv279_fwd0 A INNER JOIN Conv283_fwd0 B
		on A.batch_id=B.batch_id and A.kernel_id=B.kernel_id
		and A.tuple_id=B.tuple_id
	),
Relu285_fwd0 AS (
	select batch_id, kernel_id, tuple_id, GREATEST(value, 0) as value
		FROM Add284_fwd0 F
	),
Conv286_fwd0 AS (
	select batch_id, K.kernel_id, F.tuple_id as tuple_id,
	sum(F.value*K.value) as value
		FROM Relu285_fwd0 F INNER JOIN Conv286_weight K
		on F.kernel_id=K.order_id
	GROUP BY F.batch_id, K.kernel_id, F.tuple_id
	),
Relu287_fwd0 AS (
	select batch_id, kernel_id, tuple_id, GREATEST(value, 0) as value
		FROM Conv286_fwd0 F
	),
Relu287_fwd0_mapped AS (
	select batch_id, gn, matrix_id, order_id, value
		FROM Relu287_fwd0 F INNER JOIN Relu287_fwd0_mapping M
		on F.kernel_id = M.kernel_id and F.tuple_id = M.tuple_id
	),
Conv288_fwd0 AS (
	select batch_id, (F.gn*16+kernel_id) as kernel_id, F.matrix_id as tuple_id,
	sum(F.value*K.value) as value
		FROM Relu287_fwd0_mapped F INNER JOIN Conv288_weight K
		on F.order_id=K.order_id and F.gn=K.gn
	GROUP BY F.batch_id, F.gn, K.kernel_id, F.matrix_id
	),
Relu289_fwd0 AS (
	select batch_id, kernel_id, tuple_id, GREATEST(value, 0) as value
		FROM Conv288_fwd0 F
	),
Conv290_fwd0 AS (
	select batch_id, K.kernel_id, F.tuple_id as tuple_id,
	sum(F.value*K.value) as value
		FROM Relu289_fwd0 F INNER JOIN Conv290_weight K
		on F.kernel_id=K.order_id
	GROUP BY F.batch_id, K.kernel_id, F.tuple_id
	),
Add291_fwd0 AS (
	select A.batch_id, A.kernel_id, A.tuple_id,
	A.value + B.value as value
		FROM Relu285_fwd0 A INNER JOIN Conv290_fwd0 B
		on A.batch_id=B.batch_id and A.kernel_id=B.kernel_id
		and A.tuple_id=B.tuple_id
	),
Relu292_fwd0 AS (
	select batch_id, kernel_id, tuple_id, GREATEST(value, 0) as value
		FROM Add291_fwd0 F
	),
Conv293_fwd0 AS (
	select batch_id, K.kernel_id, F.tuple_id as tuple_id,
	sum(F.value*K.value) as value
		FROM Relu292_fwd0 F INNER JOIN Conv293_weight K
		on F.kernel_id=K.order_id
	GROUP BY F.batch_id, K.kernel_id, F.tuple_id
	),
Relu294_fwd0 AS (
	select batch_id, kernel_id, tuple_id, GREATEST(value, 0) as value
		FROM Conv293_fwd0 F
	),
Relu294_fwd0_mapped AS (
	select batch_id, gn, matrix_id, order_id, value
		FROM Relu294_fwd0 F INNER JOIN Relu294_fwd0_mapping M
		on F.kernel_id = M.kernel_id and F.tuple_id = M.tuple_id
	),
Conv295_fwd0 AS (
	select batch_id, (F.gn*16+kernel_id) as kernel_id, F.matrix_id as tuple_id,
	sum(F.value*K.value) as value
		FROM Relu294_fwd0_mapped F INNER JOIN Conv295_weight K
		on F.order_id=K.order_id and F.gn=K.gn
	GROUP BY F.batch_id, F.gn, K.kernel_id, F.matrix_id
	),
Relu296_fwd0 AS (
	select batch_id, kernel_id, tuple_id, GREATEST(value, 0) as value
		FROM Conv295_fwd0 F
	),
Conv297_fwd0 AS (
	select batch_id, K.kernel_id, F.tuple_id as tuple_id,
	sum(F.value*K.value) as value
		FROM Relu296_fwd0 F INNER JOIN Conv297_weight K
		on F.kernel_id=K.order_id
	GROUP BY F.batch_id, K.kernel_id, F.tuple_id
	),
Add298_fwd0 AS (
	select A.batch_id, A.kernel_id, A.tuple_id,
	A.value + B.value as value
		FROM Relu292_fwd0 A INNER JOIN Conv297_fwd0 B
		on A.batch_id=B.batch_id and A.kernel_id=B.kernel_id
		and A.tuple_id=B.tuple_id
	),
Relu299_fwd0 AS (
	select batch_id, kernel_id, tuple_id, GREATEST(value, 0) as value
		FROM Add298_fwd0 F
	),
Conv300_fwd0 AS (
	select batch_id, K.kernel_id, F.tuple_id as tuple_id,
	sum(F.value*K.value) as value
		FROM Relu299_fwd0 F INNER JOIN Conv300_weight K
		on F.kernel_id=K.order_id
	GROUP BY F.batch_id, K.kernel_id, F.tuple_id
	),
Relu301_fwd0 AS (
	select batch_id, kernel_id, tuple_id, GREATEST(value, 0) as value
		FROM Conv300_fwd0 F
	),
Relu301_fwd0_mapped AS (
	select batch_id, gn, matrix_id, order_id, value
		FROM Relu301_fwd0 F INNER JOIN Relu301_fwd0_mapping M
		on F.kernel_id = M.kernel_id and F.tuple_id = M.tuple_id
	),
Conv302_fwd0 AS (
	select batch_id, (F.gn*16+kernel_id) as kernel_id, F.matrix_id as tuple_id,
	sum(F.value*K.value) as value
		FROM Relu301_fwd0_mapped F INNER JOIN Conv302_weight K
		on F.order_id=K.order_id and F.gn=K.gn
	GROUP BY F.batch_id, F.gn, K.kernel_id, F.matrix_id
	),
Relu303_fwd0 AS (
	select batch_id, kernel_id, tuple_id, GREATEST(value, 0) as value
		FROM Conv302_fwd0 F
	),
Conv304_fwd0 AS (
	select batch_id, K.kernel_id, F.tuple_id as tuple_id,
	sum(F.value*K.value) as value
		FROM Relu303_fwd0 F INNER JOIN Conv304_weight K
		on F.kernel_id=K.order_id
	GROUP BY F.batch_id, K.kernel_id, F.tuple_id
	),
Add305_fwd0 AS (
	select A.batch_id, A.kernel_id, A.tuple_id,
	A.value + B.value as value
		FROM Relu299_fwd0 A INNER JOIN Conv304_fwd0 B
		on A.batch_id=B.batch_id and A.kernel_id=B.kernel_id
		and A.tuple_id=B.tuple_id
	),
Relu306_fwd0 AS (
	select batch_id, kernel_id, tuple_id, GREATEST(value, 0) as value
		FROM Add305_fwd0 F
	),
Conv307_fwd0 AS (
	select batch_id, K.kernel_id, F.tuple_id as tuple_id,
	sum(F.value*K.value) as value
		FROM Relu306_fwd0 F INNER JOIN Conv307_weight K
		on F.kernel_id=K.order_id
	GROUP BY F.batch_id, K.kernel_id, F.tuple_id
	),
Relu308_fwd0 AS (
	select batch_id, kernel_id, tuple_id, GREATEST(value, 0) as value
		FROM Conv307_fwd0 F
	),
Relu308_fwd0_mapped AS (
	select batch_id, gn, matrix_id, order_id, value
		FROM Relu308_fwd0 F INNER JOIN Relu308_fwd0_mapping M
		on F.kernel_id = M.kernel_id and F.tuple_id = M.tuple_id
	),
Conv309_fwd0 AS (
	select batch_id, (F.gn*16+kernel_id) as kernel_id, F.matrix_id as tuple_id,
	sum(F.value*K.value) as value
		FROM Relu308_fwd0_mapped F INNER JOIN Conv309_weight K
		on F.order_id=K.order_id and F.gn=K.gn
	GROUP BY F.batch_id, F.gn, K.kernel_id, F.matrix_id
	),
Relu310_fwd0 AS (
	select batch_id, kernel_id, tuple_id, GREATEST(value, 0) as value
		FROM Conv309_fwd0 F
	),
Conv311_fwd0 AS (
	select batch_id, K.kernel_id, F.tuple_id as tuple_id,
	sum(F.value*K.value) as value
		FROM Relu310_fwd0 F INNER JOIN Conv311_weight K
		on F.kernel_id=K.order_id
	GROUP BY F.batch_id, K.kernel_id, F.tuple_id
	),
Add312_fwd0 AS (
	select A.batch_id, A.kernel_id, A.tuple_id,
	A.value + B.value as value
		FROM Relu306_fwd0 A INNER JOIN Conv311_fwd0 B
		on A.batch_id=B.batch_id and A.kernel_id=B.kernel_id
		and A.tuple_id=B.tuple_id
	),
Relu313_fwd0 AS (
	select batch_id, kernel_id, tuple_id, GREATEST(value, 0) as value
		FROM Add312_fwd0 F
	),
AveragePool315_fwd0 AS (
	select batch_id, kernel_id, avg(value) as value
		FROM Relu313_fwd0
	GROUP BY batch_id, kernel_id
	),
MatMul318_fwd0 AS (
	select batch_id, K.kernel_id,
	sum(F.value*K.value) as value
		FROM AveragePool315_fwd0 F INNER JOIN Transpose317_input K
		on F.kernel_id=K.order_id
	GROUP BY F.batch_id, K.kernel_id
	),
SELECT l.name AS res FROM
cifar10_labels l
JOIN (
	SELECT DISTINCT on (batch_id) batch_id, kernel_id+1 AS label
	FROM MatMul318_fwd0 ORDER BY batch_id, value DESC) t
	ON t.label = l.label
'''

'\nWITH\nConv201_fwd0 AS (\n\tselect batch_id, K.kernel_id, F.matrix_id as tuple_id,\n\tsum(F.value*K.value) as value\n\t\tFROM input_feature_map F INNER JOIN Conv201_weight K\n\t\ton F.order_id=K.order_id\n\tGROUP BY F.batch_id, K.kernel_id, F.matrix_id\n\t),\nRelu203_fwd0 AS (\n\tselect batch_id, kernel_id, tuple_id, GREATEST(value, 0) as value\n\t\tFROM Conv201_fwd0 F\n\t),\nRelu203_fwd0_mapped AS (\n\tselect batch_id, F.kernel_id, matrix_id, order_id, value\n\t\tFROM Relu203_fwd0 F INNER JOIN Relu203_fwd0_mapping M\n\t\ton F.kernel_id = M.kernel_id and F.tuple_id = M.tuple_id\n\t),\nMaxPool205_fwd0 AS (\n\tselect batch_id, kernel_id, matrix_id as tuple_id, max(value) as value\n\t\tFROM Relu203_fwd0_mapped\n\tGROUP BY batch_id, kernel_id, matrix_id\n\t),\nConv206_fwd0 AS (\n\tselect batch_id, K.kernel_id, F.tuple_id as tuple_id,\n\tsum(F.value*K.value) as value\n\t\tFROM MaxPool205_fwd0 F INNER JOIN Conv206_weight K\n\t\ton F.kernel_id=K.order_id\n\tGROUP BY F.batch_id, K.kernel_id,